# Merge Refinitiv & Trucost database
## Import libraries

In [93]:
import pandas as pd
import numpy as np

## Functions

In [94]:
# Function to rename a string named 'FY0' to '2022', 'FY-1' to '2021', 'FY-2' to '2020', etc.
def rename_fiscal_year(fiscal_year):
    current_year = 2022
    if fiscal_year == 'FY0':
        return str(current_year)
    elif fiscal_year.startswith('FY-'):
        year_offset = int(fiscal_year[3:])
        renamed_year = current_year - year_offset
        return str(renamed_year)
    else:
        return fiscal_year
    

## Refinitiv controls
### Import data

Put the first columns about the companies in this order
![Index columns order](assets/index_order.png)

In [95]:
# Importing data from Refinitiv.
# Has 2 levels of column names and 6 index to decribe a company.
refinitiv_controls = pd.read_excel('data/Refinitiv_ Controls.xlsx', sheet_name="All controls_Agri", header=[0,1], index_col=[0,1,2,3,4,5])
# Rename the index because they disappeared.
# refinitiv_controls.index.names=['Identifier (RIC)','Company Name','Ticker Symbol','Date Became Public','NAICS Industry Group Name','Business Description']

### Clean data

In [96]:
# Remove SD column
refinitiv_controls.drop('Earnings Per Share - Standard Deviation\n(USD)\nIn the last 18 Y',axis=1,inplace=True, level=0)
# Rename the column names based on fiscal years
refinitiv_controls.rename(columns=rename_fiscal_year, inplace=True, level=1)
# Remove year 2022
refinitiv_controls.drop('2022',axis=1,inplace=True, level=1)


### Reshape data

In [97]:
refinitiv_controls_stacked = refinitiv_controls.stack(level=1, dropna=False)

## Refinitiv returns
### Import data

It's necesserry to add year/month in the column header for returns per month
![Change in columns header](assets\returns_month.png)

In [98]:
refinitiv_returns = pd.read_excel('data/Refinitiv_ Returns.xlsx', sheet_name="Agriculture Comp, Full + Ticker", header=[0,1,2], index_col=[0,1,2,3,4,5])

### Clean data

In [99]:
# Remove return per year columns
refinitiv_returns_month = refinitiv_returns.drop('Total Return by Year (01.01.2004-31.12.2022)',axis=1,level=0)
# Remove year 2004, 2022
refinitiv_returns_month.drop([2004,2022],axis=1,inplace=True, level=1)

### Generate yearly returns statistics

In [100]:
# Gerenate column with available fiscal months per year
refinitiv_returns_count = refinitiv_returns_month.groupby(axis=1, level=1).count()
# Add header name
refinitiv_returns_count = pd.concat([refinitiv_returns_count], axis=1, keys=["Available fiscal months"])

In [101]:
# Generate mean for each year
refinitiv_returns_mean = refinitiv_returns_month.groupby(axis=1, level=1).mean()
# Add header name
refinitiv_returns_mean = pd.concat([refinitiv_returns_mean], axis=1, keys=["Mean"])


In [102]:
# Generate composed return for each year
refinitiv_returns_composed = (refinitiv_returns_month+1).groupby(axis=1, level=1).prod()-1
# Add header name
refinitiv_returns_composed = pd.concat([refinitiv_returns_composed], axis=1, keys=["Composed Return"])


In [103]:
# Generate Standard Deviation for each year
refinitiv_returns_std = refinitiv_returns_month.groupby(axis=1, level=1).std()
# Add header name
refinitiv_returns_std = pd.concat([refinitiv_returns_std], axis=1, keys=["Standard Deviation"])

In [104]:
# Put every result in the same dataframe
refinitiv_returns_year = pd.concat([refinitiv_returns_count, refinitiv_returns_mean, refinitiv_returns_composed, refinitiv_returns_std], axis=1)


### Reshape data

In [105]:
refinitiv_returns_year_stacked = refinitiv_returns_year.stack(level=1, dropna=False)

## Refinitiv Beta
TODO

## Merge Refinitiv datas

TODO

## Trucost
### Import data

In [158]:
trucost = pd.read_excel('data/Trucost-2005-2021.xlsx', sheet_name="kglwvfc8zch6f3ps", header=[0])

### Clean data

In [167]:
# Drop empty Tickers
trucost.dropna(subset='Ticker',inplac=True)

,Institution ID,Period End Date,Intensity: GHG Scope 1,Intensity: GHG Scope 3 Upstream,Intensity: GHG Scope 2,Absolute: GHG Direct,Absolute: GHG Scope 1,Absolute: GHG Scope 2,Absolute: GHG Scope 3 Upstream,Absolute: Natural Resources Direct & Indirect Cost,Impact Ratio: Natural Resources Dir & Ind Cost,Intensity: Water Direct and Purchased,Absolute: Water Direct and Purchased,Intensity: Waste Landfill Direct and Indirect,Absolute: Waste Landfill Direct and Indirect,Intensity: GHG Scope 3 Downstream,Absolute: GHG Scope 3 Downstream,Ticker
0,102926,2017-09-30,13.580337,33.677460,63.912690,1418.832836,1418.832836,6677.406104,3518.519981,0.043495,0.041631,960.357659,1.003353e+05,4.215918,440.466431,388.125330,40550.170150,BRT
1,102926,2018-09-30,12.994718,32.708122,62.973481,1544.708106,1544.708106,7485.783680,3888.079888,0.051594,0.043403,917.031892,1.090094e+05,4.046544,481.020828,49.994736,5942.974302,BRT
2,102926,2019-12-31,12.880980,30.416402,57.184705,347.902377,347.902377,1544.501711,821.516604,0.010716,0.039676,890.481587,2.405102e+04,3.984781,107.624955,54.530439,1472.812632,BRT
3,102926,2020-12-31,12.648000,30.448000,55.493000,355.443000,355.443000,1559.456000,855.640000,0.011000,0.039000,870.169000,2.445349e+04,3.930470,110.454073,52.372005,1471.758087,BRT
4,102926,2021-12-31,12.027000,31.364000,58.632000,390.259000,390.259000,1902.537000,1017.734000,0.013000,0.041000,818.428000,2.655718e+04,3.760218,122.015304,58.871538,1910.322548,BRT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3093,27762653,2020-12-31,16.912000,197.563000,15.784000,4194.712000,4171.127000,3893.030000,48726.307000,0.124000,0.050000,6960.104000,1.716616e+06,11.206538,2763.942124,1097.132391,270592.978900,6601
3094,28714179,2020-12-31,16.070000,89.377000,11.927000,723.792000,718.704000,533.404000,3997.269000,0.012000,0.027000,11172.433000,4.996703e+05,6.723700,300.707377,9.646249,431.414000,APPH
3095,28714179,2021-12-31,15.087000,93.468000,12.601000,919.685000,913.219000,762.746000,5657.501000,0.018000,0.029000,10489.654000,6.349283e+05,6.445710,390.152356,NaN,NaN,APPH
3096,29427292,2020-12-31,4.349000,35.285000,5.170000,229.464000,229.464000,272.735000,1861.521000,0.007000,0.012000,40.609000,2.142378e+03,5.990554,316.040665,101.891596,5375.444008,BFG
